# RAG Assignment: Baseline (Bi-Encoder) vs Re-Ranking (Cross-Encoder)
**Document:** IIIT Hyderabad Students' Parliament Annual Report 2025–2026 (95 pages)

## Step 0: Environment setup
Install libraries, check the GPU and input files, and connect to the Gemini API.

In [6]:
# Install the libraries.
# google-genai          -> official Gemini API client (the LLM that writes answers)
# pymupdf               -> reads text out of the PDF
# faiss-cpu             -> vector store for the bi-encoder embeddings
# sentence-transformers -> bi-encoder and cross-encoder models
!pip install -q -U google-genai
!pip install -q pymupdf faiss-cpu sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 69.6 MB/s eta 0:00:00:00:0

In [7]:
import os
import time
from importlib.metadata import version

import numpy as np
import torch
import pymupdf                    # PDF reader (newer name for "fitz")
import faiss                      # vector store
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from kaggle_secrets import UserSecretsClient

# Library versions: record these for reproducibility
for pkg in ["torch", "sentence-transformers", "faiss-cpu", "pymupdf", "google-genai"]:
    print(f"{pkg:22s} {version(pkg)}")

# Hardware: matters for the latency table later
print("\nGPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# List every attached input file so we know the exact PDF path
print()
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print("Input file:", os.path.join(root, f))

torch                  2.11.0+cu128
sentence-transformers  5.7.0
faiss-cpu              1.15.1
pymupdf                1.28.2
google-genai           2.31.0

GPU available: True
GPU name: Tesla T4

Input file: /kaggle/input/datasets/narsingmadaka/parliament-report-2025-26/parliament-activities-report-annual-2025-2026.pdf


## Step 1: Load the PDF and clean the text
- Skip PDF pages 1–5 (cover + table of contents): they contain no answers but match many queries.
- Remove the running header, chapter title and page number repeated on every page.
- Remove soft hyphens (U+00AD) that split words across lines.
- Keep the PDF page number with each page's text, so every chunk can later be traced to its page.

In [8]:
import re

PDF_PATH = "/kaggle/input/datasets/narsingmadaka/parliament-report-2025-26/parliament-activities-report-annual-2025-2026.pdf"

doc = pymupdf.open(PDF_PATH)
print("Total pages:", doc.page_count)

HEADER = "Annual Report by the Students’ Parliament 2025–2026"   # repeated at the top of every page
SKIP_PAGES = 5                                                  # PDF pages 1–5 = cover + contents

def clean_page(raw_text):
    """Clean the raw text of one PDF page."""
    lines = [line.strip() for line in raw_text.split("\n")]
    lines = [line for line in lines if line]          # drop empty lines

    # Line 0 = running header, line 1 = running chapter title -> remove both
    if lines and lines[0] == HEADER:
        lines = lines[2:]
    # Last line = page number -> remove it
    if lines and lines[-1].isdigit():
        lines = lines[:-1]

    text = "\n".join(lines)
    text = text.replace("\u00ad\n", "").replace("\u00ad", "")   # soft hyphen: "Interna-\ntional" -> "International"
    text = text.replace("-\n", "-")                             # real hyphen at line end: "in-\ncharge" -> "in-charge"
    text = re.sub(r"\s+", " ", text).strip()                    # collapse all whitespace into single spaces
    return text

# Keep each page's cleaned text together with its PDF page number (1-based)
pages = []
for i in range(SKIP_PAGES, doc.page_count):
    pages.append({"page": i + 1, "text": clean_page(doc[i].get_text())})

print("Pages kept:", len(pages))
print("Total words:", sum(len(p["text"].split()) for p in pages))
print("Soft hyphens left:", sum(p["text"].count("\u00ad") for p in pages))
print("Header lines left:", sum(p["text"].count(HEADER) for p in pages))

# Look at one cleaned page: PDF page 18 contains Section 3.6 (Attendance and Leave Rules)
sample = next(p for p in pages if p["page"] == 18)
print(f"\n--- PDF page {sample['page']} ---\n{sample['text']}")

Total pages: 95
Pages kept: 90
Total words: 31868
Soft hyphens left: 0
Header lines left: 0

--- PDF page 18 ---
them to fill in the forms and stating that the feedback is anonymous and is considered by the Institute. October 10, 2025 The Academic Office shared participation statistics with the Parliament. In Monsoon 2025, of 98 full-semester courses, 2 had a participation rate above 50% and 43 had a participation rate below 10%. November 2025 The Council confirmed with the Academic Office that feedback responses are not linked to the student who submits them. In Progress Participation remains low. The Council began discussing a separate feedback mechanism for Teaching Assistants. At the start of the term the Council set the aim of having course feedback results released publicly, and the Institute did not agree to this during the term. 3.4 Feedback Form for First-Year Undergraduate Students First-year undergraduate students reported that they had to fill in the course feedback form fo

## Step 2: Chunking
Split the cleaned text into fixed-size windows of 150 words with a 30-word overlap.
- 150 words ≈ 194 tokens on average (max 253), under the bi-encoder's 256-token limit, so nothing is truncated.
- The overlap reduces the chance of a sentence being lost at a chunk boundary.
- Each chunk keeps its id and its PDF page range. Both pipelines use exactly these chunks.

In [9]:
CHUNK_SIZE = 150   # words per chunk
OVERLAP = 30       # words shared by consecutive chunks
STEP = CHUNK_SIZE - OVERLAP

# 1) Join all pages into one word list, remembering which PDF page each word came from
words, word_pages = [], []
for p in pages:
    page_words = p["text"].split()
    words.extend(page_words)
    word_pages.extend([p["page"]] * len(page_words))

# 2) Slide a 150-word window forward by 120 words at a time
chunks = []
for start in range(0, len(words), STEP):
    end = min(start + CHUNK_SIZE, len(words))
    chunks.append({
        "chunk_id": len(chunks),
        "text": " ".join(words[start:end]),
        "page_start": word_pages[start],
        "page_end": word_pages[end - 1],
    })
    if end == len(words):      # reached the end of the document
        break

# 3) Sanity checks
lengths = [len(c["text"].split()) for c in chunks]
print("Total words:", len(words))
print("Number of chunks:", len(chunks))
print("Words per chunk: min", min(lengths), "| avg", round(sum(lengths) / len(lengths), 1), "| max", max(lengths))

overlap_ok = all(
    chunks[i]["text"].split()[-OVERLAP:] == chunks[i + 1]["text"].split()[:OVERLAP]
    for i in range(len(chunks) - 2)
)
print("Overlap check (last 30 words of each chunk = first 30 of next):", overlap_ok)

# 4) Look at a few chunks: the first, the one holding Section 3.6, and the last
hits = [c["chunk_id"] for c in chunks if "3.6 Attendance and Leave Rules" in c["text"]]
print("Chunks containing '3.6 Attendance and Leave Rules':", hits)

for cid in [0, hits[0], len(chunks) - 1]:
    c = chunks[cid]
    print(f"\n--- Chunk {cid} | pages {c['page_start']}-{c['page_end']} | {len(c['text'].split())} words ---")
    print(c["text"])

Total words: 31868
Number of chunks: 266
Words per chunk: min 68 | avg 149.7 | max 150
Overlap check (last 30 words of each chunk = first 30 of next): True
Chunks containing '3.6 Attendance and Leave Rules': [39]

--- Chunk 0 | pages 6-6 | 150 words ---
1 Introduction 1.1 The Students’ Parliament The Students’ Parliament is the elected body that represents the students of the International Institute of Information Technology, Hyderabad, to the Institute administration. It traces its origins to the early 2000s and is one of the oldest student bodies at the Institute. Students elect the members each year by secret ballot, using the single transferable vote, with one member for every 50 students. The Speaker administers the Parliament internally, and the General Secretary is its point of contact with the Institute. The Parliament works through seven councils, namely the Academic, Research, Health, Hostel, Infrastructure, Mess, and Stall Councils. Each council has a Council Secretary and t

## Step 3: Bi-encoder embeddings and FAISS vector store
- Model: `sentence-transformers/all-MiniLM-L6-v2` (384-dim vectors, 256-token limit).
- Every chunk is embedded once (offline indexing). Vectors are L2-normalised, so inner product = cosine similarity.
- Index: FAISS `IndexFlatIP`, exact (brute-force) inner-product search over all chunks.

In [10]:
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1) Load the bi-encoder
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=device)
print("Bi-encoder:", BI_ENCODER_NAME, "| device:", device)
print("Max input tokens:", bi_encoder.max_seq_length)

# 2) Confirm no chunk is longer than the model can read (otherwise it would be silently truncated)
token_counts = [len(bi_encoder.tokenizer(c["text"])["input_ids"]) for c in chunks]
print("Tokens per chunk: max", max(token_counts),
      "| chunks over limit:", sum(t > bi_encoder.max_seq_length for t in token_counts))

# 3) Embed every chunk once (offline indexing step)
chunk_texts = [c["text"] for c in chunks]
bi_encoder.encode(["warm-up"], show_progress_bar=False)   # first GPU call is slow; keep it out of the timing

t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode(
    chunk_texts,
    batch_size=64,
    normalize_embeddings=True,     # unit-length vectors -> inner product == cosine similarity
    convert_to_numpy=True,
    show_progress_bar=False,
).astype("float32")
embed_time = time.perf_counter() - t0

# 4) Put all vectors into a FAISS index (exact inner-product search)
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Embeddings shape:", chunk_embeddings.shape)
print("Vectors in FAISS index:", index.ntotal)
print(f"Vector lengths: min {norms.min():.4f} | max {norms.max():.4f}")
print(f"Time to embed all {len(chunks)} chunks (one-time): {embed_time:.2f} s")

# 5) Sanity check: one search, top 3
query = "What is the maximum amount the Student Medical Reimbursement Fund pays per person?"
q_emb = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
scores, ids = index.search(q_emb, 3)

print("\nQuery:", query)
for rank, (cid, score) in enumerate(zip(ids[0], scores[0]), start=1):
    c = chunks[cid]
    print(f"  #{rank} chunk {cid} (p{c['page_start']}-{c['page_end']}) score {score:.4f} | {c['text'][:100]}...")
print("Top-1 chunk contains the answer 'Rs. 50,000':", "50,000" in chunks[ids[0][0]]["text"])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Bi-encoder: sentence-transformers/all-MiniLM-L6-v2 | device: cuda
Max input tokens: 256
Tokens per chunk: max 253 | chunks over limit: 0
Embeddings shape: (266, 384)
Vectors in FAISS index: 266
Vector lengths: min 1.0000 | max 1.0000
Time to embed all 266 chunks (one-time): 0.89 s

Query: What is the maximum amount the Student Medical Reimbursement Fund pays per person?
  #1 chunk 95 (p37-37) score 0.5438 | rooms, recorded in Section 5.11, found wheelchairs without footrests. In Progress CHMC is considerin...
  #2 chunk 49 (p21-22) score 0.3811 | Assistant from Monsoon 2026. 2. A monthly stipend of Rs. 15,000 for a Half Teaching Assistant. 3. A ...
  #3 chunk 22 (p13-13) score 0.3802 | a preparation update, a post-event report, and a financial report. A student or student body that ba...
Top-1 chunk contains the answer 'Rs. 50,000': True


## Step 4: Pipeline 1 (baseline retrieval)
1. Bi-encoder + FAISS retrieves the **top 10** chunks by cosine similarity.
2. The **top 3** are passed, unchanged, to Gemini using a fixed prompt template → **Answer 1**.
The same prompt template and LLM settings are reused for Pipeline 2, so only the chosen chunks differ.

In [11]:
from google.genai import types

MODEL_NAME = "gemini-2.5-flash"   # same LLM for Answer 1 and Answer 2
PAUSE = 15                        # seconds to wait between Gemini calls (free tier allows only a few per minute)

# Gemini client with a 60-second timeout, so a network problem raises an error instead of hanging forever
client = genai.Client(
    api_key=UserSecretsClient().get_secret("GEMINI_API_KEY"),
    http_options=types.HttpOptions(timeout=60_000),             # milliseconds
)

GEN_CONFIG = types.GenerateContentConfig(
    temperature=0.0,                                            # as deterministic as possible
    thinking_config=types.ThinkingConfig(thinking_budget=0),    # no hidden "thinking": faster, more consistent timings
)

# ONE prompt template, used unchanged by both pipelines (only the context chunks differ)
PROMPT_TEMPLATE = """You are answering questions about the IIIT Hyderabad Students' Parliament Annual Report 2025-2026.
Use ONLY the context passages below. Do not use outside knowledge.
If the passages do not contain the answer, reply exactly: "The provided context does not contain this information."
Answer concisely and cite the passage numbers you used, like [1] or [2].

Context:
{context}

Question: {question}
Answer:"""

def generate_answer(prompt, retries=3):
    """Send the prompt to Gemini. Stop at once if the DAILY quota is used up; retry on per-minute limits."""
    for attempt in range(1, retries + 1):
        try:
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt, config=GEN_CONFIG)
            return (response.text or "").strip()
        except Exception as e:
            msg = str(e)
            if "perday" in msg.lower():
                raise RuntimeError(f"Daily free quota for {MODEL_NAME} is used up. "
                                   "Change MODEL_NAME or wait for the daily reset.") from None
            print(f"  Gemini error (attempt {attempt}/{retries}): {msg[:120]}")
            if attempt == retries:
                raise
            time.sleep(30)    # per-minute limit: wait, then retry

print("LLM:", MODEL_NAME)

LLM: gemini-2.5-flash


In [12]:
TOP_K = 10   # candidates retrieved by the bi-encoder
TOP_N = 3    # chunks passed to the LLM

def retrieve(query, k=TOP_K):
    """Bi-encoder + FAISS: return the top-k (chunk_id, cosine score) pairs, best first."""
    q_emb = bi_encoder.encode([query], normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
    scores, ids = index.search(q_emb, k)
    return [(int(cid), float(s)) for cid, s in zip(ids[0], scores[0])]

def page_label(cid):
    """PDF page(s) of a chunk, e.g. '37' or '21-22'."""
    c = chunks[cid]
    return str(c["page_start"]) if c["page_start"] == c["page_end"] else f"{c['page_start']}-{c['page_end']}"

def build_prompt(query, chunk_ids):
    """Fill the shared prompt template with the chosen chunks, numbered [1], [2], [3]."""
    context = "\n\n".join(
        f"[{i}] (chunk {cid}, PDF page {page_label(cid)})\n{chunks[cid]['text']}"
        for i, cid in enumerate(chunk_ids, start=1)
    )
    return PROMPT_TEMPLATE.format(context=context, question=query)

def pipeline1(query):
    """Baseline: bi-encoder top-10 -> keep the top-3 as they are -> LLM -> Answer 1."""
    t0 = time.perf_counter()
    candidates = retrieve(query)                      # top-10 by cosine similarity
    t_retrieval = time.perf_counter() - t0

    top3 = [cid for cid, _ in candidates[:TOP_N]]     # top-3 straight from the bi-encoder ranking

    t1 = time.perf_counter()
    answer = generate_answer(build_prompt(query, top3))
    t_llm = time.perf_counter() - t1

    return {"query": query, "candidates": candidates, "top3": top3,
            "answer": answer, "t_retrieval": t_retrieval, "t_llm": t_llm}

def show_pipeline1(result):
    """Print the top-10 table, Answer 1 and the timings."""
    print("Query:", result["query"])
    print("\nBi-encoder top-10   (* = sent to the LLM)")
    print(f"{'rank':>4} {'chunk':>5} {'page':>6} {'cosine':>7}   text")
    for rank, (cid, score) in enumerate(result["candidates"], start=1):
        mark = "*" if cid in result["top3"] else " "
        print(f"{rank:>4} {cid:>5} {page_label(cid):>6} {score:7.4f} {mark} {chunks[cid]['text'][:60]}...")
    print("\nAnswer 1:", result["answer"])
    print(f"\nTime: retrieval {result['t_retrieval'] * 1000:.1f} ms | LLM {result['t_llm']:.2f} s")

In [13]:
r1 = pipeline1("What is the maximum amount the Student Medical Reimbursement Fund pays per person?")
show_pipeline1(r1)

Query: What is the maximum amount the Student Medical Reimbursement Fund pays per person?

Bi-encoder top-10   (* = sent to the LLM)
rank chunk   page  cosine   text
   1    95     37  0.5438 * rooms, recorded in Section 5.11, found wheelchairs without f...
   2    49  21-22  0.3811 * Assistant from Monsoon 2026. 2. A monthly stipend of Rs. 15,...
   3    22     13  0.3802 * a preparation update, a post-event report, and a financial r...
   4    45  20-21  0.3781   Full Teaching Assistant. December 22, 2025 The Council wrote...
   5    80     32  0.3677   measures in place, no further action was needed. 5.2 Grievan...
   6    53  22-23  0.3577   with the Council in advance, that the Council’s objections h...
   7    46     21  0.3571   yet begun, but would be completed in time. April 23, June 19...
   8    44     20  0.3566   on Sundays, and to extend the examination period past the en...
   9    51     22  0.3507   5 hours a week for a Quarter, 10 for a Half, and 20 for a Fu...
  10  

In [14]:
print(build_prompt(r1["query"], r1["top3"]))

You are answering questions about the IIIT Hyderabad Students' Parliament Annual Report 2025-2026.
Use ONLY the context passages below. Do not use outside knowledge.
If the passages do not contain the answer, reply exactly: "The provided context does not contain this information."
Answer concisely and cite the passage numbers you used, like [1] or [2].

Context:
[1] (chunk 95, PDF page 37)
rooms, recorded in Section 5.11, found wheelchairs without footrests. In Progress CHMC is considering wheelchair models for the campus and the hostels, and arranging their purchase. It plans to buy orthopaedic wheelchairs, since the most common injuries among students are sprains and fractures sustained during sports. The faculty and staff quarters have different needs, and a suitable model is to be chosen for them. 5.13 Emergency Medical Assistance Limit The Institute previously insured students, and replaced the insurance with the Student Medical Reimbursement Fund, which pays up to Rs. 50,000 per 

## Step 5: Pipeline 2 (cross-encoder re-ranking)
1. Take the same top-10 candidates from the bi-encoder.
2. Score all 10 (query, chunk) pairs with `cross-encoder/ms-marco-MiniLM-L-6-v2`, which reads the query and chunk together.
3. Re-sort by cross-encoder score and pass the new top-3 to Gemini with the identical prompt → **Answer 2**.
The table shows each chunk's old (bi-encoder) rank and new (cross-encoder) rank.

In [15]:
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# 1) Load the cross-encoder (reads query + chunk TOGETHER, max 512 tokens for the pair)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=device, max_length=512)
cross_encoder.predict([("warm-up query", "warm-up passage")], show_progress_bar=False)   # first GPU call is slow
print("Cross-encoder:", CROSS_ENCODER_NAME, "| device:", device)

# 2) Check that (query + chunk) pairs fit in 512 tokens, using a typical question with every chunk
test_q = "What is the maximum amount the Student Medical Reimbursement Fund pays per person?"
pair_tokens = [len(cross_encoder.tokenizer(test_q, c["text"])["input_ids"]) for c in chunks]
print("Longest (query + chunk) pair:", max(pair_tokens), "tokens (limit 512)")

def rerank(query, candidates):
    """Score each (query, chunk) pair jointly with the cross-encoder and sort best-first.
    Returns a list of (chunk_id, cross-encoder score, original bi-encoder rank)."""
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs, batch_size=len(pairs), show_progress_bar=False)
    bi_rank = {cid: rank for rank, (cid, _) in enumerate(candidates, start=1)}
    scored = [(cid, float(s), bi_rank[cid]) for (cid, _), s in zip(candidates, ce_scores)]
    return sorted(scored, key=lambda x: x[1], reverse=True)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 | device: cuda
Longest (query + chunk) pair: 271 tokens (limit 512)


In [16]:
def pipeline2(query):
    """Re-ranking: bi-encoder top-10 -> cross-encoder re-sorts them -> top-3 -> same prompt -> Answer 2."""
    t0 = time.perf_counter()
    candidates = retrieve(query)                          # the SAME top-10 as Pipeline 1
    t_retrieval = time.perf_counter() - t0

    t1 = time.perf_counter()
    reranked = rerank(query, candidates)                  # cross-encoder scores all 10 pairs
    t_rerank = time.perf_counter() - t1

    top3 = [cid for cid, _, _ in reranked[:TOP_N]]        # top-3 after re-ranking

    t2 = time.perf_counter()
    answer = generate_answer(build_prompt(query, top3))   # identical prompt template as Pipeline 1
    t_llm = time.perf_counter() - t2

    return {"query": query, "candidates": candidates, "reranked": reranked, "top3": top3,
            "answer": answer, "t_retrieval": t_retrieval, "t_rerank": t_rerank, "t_llm": t_llm}

def show_pipeline2(result):
    """Print the re-ranked table (with each chunk's old bi-encoder rank), Answer 2 and the timings."""
    print("Query:", result["query"])
    print("\nCross-encoder re-ranking of the same top-10   (* = sent to the LLM, shift + = moved up)")
    print(f"{'new':>4} {'old':>4} {'shift':>5} {'chunk':>5} {'page':>6} {'CE score':>8}   text")
    for new_rank, (cid, score, old_rank) in enumerate(result["reranked"], start=1):
        shift = old_rank - new_rank
        shift_txt = f"+{shift}" if shift > 0 else (str(shift) if shift < 0 else "=")
        mark = "*" if cid in result["top3"] else " "
        print(f"{new_rank:>4} {old_rank:>4} {shift_txt:>5} {cid:>5} {page_label(cid):>6} {score:8.3f} {mark} {chunks[cid]['text'][:50]}...")
    print("\nAnswer 2:", result["answer"])
    print(f"\nTime: retrieval {result['t_retrieval'] * 1000:.1f} ms | re-rank {result['t_rerank'] * 1000:.1f} ms"
          f" | LLM {result['t_llm']:.2f} s")

In [17]:
time.sleep(PAUSE)   # space out Gemini calls (free-tier rate limit); not part of any timing
r2 = pipeline2(r1["query"])
show_pipeline2(r2)

print("\nSame top-10 candidates as Pipeline 1:", [c for c, _ in r2["candidates"]] == [c for c, _ in r1["candidates"]])
print("Top-3 Pipeline 1:", r1["top3"], "| Top-3 Pipeline 2:", r2["top3"])

Query: What is the maximum amount the Student Medical Reimbursement Fund pays per person?

Cross-encoder re-ranking of the same top-10   (* = sent to the LLM, shift + = moved up)
 new  old shift chunk   page CE score   text
   1    1     =    95     37    4.459 * rooms, recorded in Section 5.11, found wheelchairs...
   2    4    +2    45  20-21   -9.062 * Full Teaching Assistant. December 22, 2025 The Cou...
   3   10    +7    84     33   -9.421 * agreement with Care Hospital remained open at the ...
   4    2    -2    49  21-22   -9.783   Assistant from Monsoon 2026. 2. A monthly stipend ...
   5    9    +4    51     22   -9.965   5 hours a week for a Quarter, 10 for a Half, and 2...
   6    5    -1    80     32  -10.295   measures in place, no further action was needed. 5...
   7    8    +1    44     20  -10.619   on Sundays, and to extend the examination period p...
   8    7    -1    46     21  -10.776   yet begun, but would be completed in time. April 2...
   9    6    -3    53  2

In [18]:
q = "How many confirmed cases of Hand, Foot, and Mouth disease were recorded?"
time.sleep(PAUSE)
r1_hfmd = pipeline1(q)
show_pipeline1(r1_hfmd)
print("\n" + "-" * 90 + "\n")
time.sleep(PAUSE)
r2_hfmd = pipeline2(q)
show_pipeline2(r2_hfmd)

Query: How many confirmed cases of Hand, Foot, and Mouth disease were recorded?

Bi-encoder top-10   (* = sent to the LLM)
rank chunk   page  cosine   text
   1    97  37-38  0.6125 * which did not happen. August 12 to 18, 2026 Four further con...
   2    98     38  0.5576 * it spreads quickly through contact with surfaces, saliva, re...
   3     4      7  0.5386 * orientation meeting (Section 3.13). • In August 2026, the In...
   4    99     38  0.3604   the disease had not been recognised. With their consent, the...
   5   128     47  0.3290   in-charge, and that in-charges receive communication trainin...
   6    76     31  0.3037   and checks of the RO plants were made more frequent. The Cou...
   7    77     31  0.3013   Kadamba Nivas and from the Vindhya water coolers. September ...
   8   100     38  0.3010   to sanitise shared equipment and articles, to avoid close co...
   9   122     46  0.2919   and C. April 14, 2026 The damaged municipal pipeline supplyi...
  10    96     3

## Step 6: Failure-mode analysis (stress testing)
For each query, the **gold chunk** is the chunk containing the true answer. It is found by searching for the exact wording from the PDF, so it can be checked against the page number.
We compare where the gold chunk lands in each pipeline and whether it reaches the top 3 sent to the LLM, then compare Answer 1 with Answer 2.

In [19]:
import json

failure_log = []   # one row per test query; used for the summary tables later
LOG_FILE = "/kaggle/working/failure_log.json"   # results are also saved here after every query

def find_gold(phrase):
    """Ids of the chunks that contain the exact wording of the true answer (copied from the PDF)."""
    return [c["chunk_id"] for c in chunks if phrase in c["text"]]

def compare(category, query, gold_phrase, true_answer):
    """Run both pipelines on one query and show where the gold chunk lands in each."""
    gold = find_gold(gold_phrase)
    time.sleep(PAUSE)      # space out Gemini calls (free-tier rate limit); not part of any timing
    r1 = pipeline1(query)
    time.sleep(PAUSE)
    r2 = pipeline2(query)

    full_bi_rank = {cid: r for r, (cid, _) in enumerate(retrieve(query, k=len(chunks)), start=1)}
    ce_rank = {cid: new for new, (cid, _, _) in enumerate(r2["reranked"], start=1)}
    ce_score = {cid: s for cid, s, _ in r2["reranked"]}

    print("=" * 100)
    print(f"[{category}] {query}")
    print(f"True answer (from the PDF): {true_answer}")
    print(f"Gold chunk(s): {gold}\n")

    print(f"{'chunk':>5} {'page':>6} {'bi rank':>7} {'cosine':>7} {'CE rank':>7} {'CE score':>8}")
    for bi_r, (cid, cos) in enumerate(r1["candidates"], start=1):
        flag = "   <- GOLD" if cid in gold else ""
        print(f"{cid:>5} {page_label(cid):>6} {bi_r:>7} {cos:7.4f} {ce_rank[cid]:>7} {ce_score[cid]:8.3f}{flag}")

    for g in gold:
        new = f"#{ce_rank[g]}" if g in ce_rank else "not re-ranked (outside the bi-encoder top-10)"
        print(f"\nGold chunk {g}: bi-encoder rank #{full_bi_rank[g]}  ->  cross-encoder rank {new}")

    in1 = bool(set(gold) & set(r1["top3"]))
    in2 = bool(set(gold) & set(r2["top3"]))
    print(f"Pipeline 1 top-3: {r1['top3']}   gold included: {'YES' if in1 else 'NO'}")
    print(f"Pipeline 2 top-3: {r2['top3']}   gold included: {'YES' if in2 else 'NO'}")

    print(f"\nAnswer 1 (baseline):  {r1['answer']}")
    print(f"Answer 2 (re-ranked): {r2['answer']}")
    print(f"\nTime  P1: retrieval {r1['t_retrieval']*1000:.1f} ms, LLM {r1['t_llm']:.2f} s"
          f"  |  P2: retrieval {r2['t_retrieval']*1000:.1f} ms, re-rank {r2['t_rerank']*1000:.1f} ms, LLM {r2['t_llm']:.2f} s")

    failure_log.append({
        "category": category, "query": query, "true_answer": true_answer, "gold": gold,
        "bi_rank": min(full_bi_rank[g] for g in gold),
        "ce_rank": min([ce_rank[g] for g in gold if g in ce_rank], default=None),
        "gold_in_p1": in1, "gold_in_p2": in2,
        "answer1": r1["answer"], "answer2": r2["answer"],
        "t_llm1": r1["t_llm"], "t_llm2": r2["t_llm"],
    })
    json.dump(failure_log, open(LOG_FILE, "w"), indent=1)   # save after every query

In [20]:
# (category, query, exact phrase of the true answer in the PDF, true answer in short)
FAILURE_QUERIES = [
    ("Keyword trap",
     "What is the maximum number of working days of leave a student can take in a semester?",
     "up to 18 working days of leave in a semester",
     "18 working days per semester (Sec 3.6, PDF p.19). The 6-day cap is only a proposal that is on hold (Sec 3.18)."),
    ("Exception (penalty)",
     "What happens to a student who backs out of an approved event without a valid reason?",
     "receives reduced support for the next event",
     "They receive reduced support for the next event they enter (draft policy, Sec 2.8 rule 5, PDF p.13)."),
    ("Exception (except-clause)",
     "How many days of leave can a student get for representing the Institute at external events?",
     "up to 7 working days of leave per semester",
     "Up to 7 working days per semester, with exceptions for events of high importance (draft policy, Sec 2.8, PDF p.12-13)."),
    ("Negation",
     "Has any water softener been installed in the hostels?",
     "no softener has been installed",
     "No. The planned trials did not take place and no softener has been installed (Sec 6.4, PDF p.45)."),
    ("Buried chunk (numbers)",
     "What is the current monthly stipend for a Quarter Teaching Assistant?",
     "Rs. 7,500 for a Quarter",
     "Rs. 7,500 per month from Monsoon 2026 (Sec 3.13, PDF p.21). Rs. 6,000 was the old amount; Rs. 8,000 was only demanded."),
    ("Limit: gold outside top-10",
     "Was weekly room cleaning introduced in the hostels?",
     "Weekly room cleaning was not introduced",
     "No, weekly room cleaning was not introduced during the term (Sec 6.1, PDF p.41)."),
]

# 2 Gemini calls per query -> 12 calls in total (compare() pauses between calls)
for category, query, phrase, truth in FAILURE_QUERIES:
    compare(category, query, phrase, truth)

[Keyword trap] What is the maximum number of working days of leave a student can take in a semester?
True answer (from the PDF): 18 working days per semester (Sec 3.6, PDF p.19). The 6-day cap is only a proposal that is on hold (Sec 3.18).
Gold chunk(s): [40]

chunk   page bi rank  cosine CE rank CE score
   60  24-25       1  0.7597       5    3.324
   62     25       2  0.6886       6    2.920
   63     25       3  0.5779       4    3.494
   61     25       4  0.5754       3    4.336
   59     24       5  0.5686       7    2.526
   21  12-13       6  0.5619       2    4.998
   40  18-19       7  0.5559       1    5.342   <- GOLD
   22     13       8  0.5323       9   -1.129
   92  35-36       9  0.5111      10   -1.545
   91     35      10  0.4680       8    0.179

Gold chunk 40: bi-encoder rank #7  ->  cross-encoder rank #1
Pipeline 1 top-3: [60, 62, 63]   gold included: NO
Pipeline 2 top-3: [40, 21, 61]   gold included: YES

Answer 1 (baseline):  The provided context does not conta

## Step 7: Summary of the failure queries
Rank of the gold chunk in each pipeline, whether it reached the top 3 sent to the LLM, and Answer 1 vs Answer 2 next to the true answer from the PDF.

In [22]:
import os

# After a session restart, reload the saved results instead of calling Gemini again
if not failure_log and os.path.exists(LOG_FILE):
    failure_log = json.load(open(LOG_FILE))

print("Rank of the gold chunk (the chunk that contains the true answer)\n")
print(f"{'category':<28} {'gold chunk':>10} {'bi-encoder':>11} {'cross-encoder':>14} {'in P1 top-3':>12} {'in P2 top-3':>12}")
for row in failure_log:
    ce = f"#{row['ce_rank']}" if row["ce_rank"] else "not in top-10"
    print(f"{row['category']:<28} {str(row['gold']):>10} {'#' + str(row['bi_rank']):>11} {ce:>14} "
          f"{'YES' if row['gold_in_p1'] else 'NO':>12} {'YES' if row['gold_in_p2'] else 'NO':>12}")

print("\n\nAnswer 1 vs Answer 2\n")
for i, row in enumerate(failure_log, start=1):
    print(f"{i}. [{row['category']}] {row['query']}")
    print(f"   True answer: {row['true_answer']}")
    print(f"   Answer 1 (baseline):  {row['answer1']}")
    print(f"   Answer 2 (re-ranked): {row['answer2']}\n")

llm_times = [row["t_llm1"] for row in failure_log] + [row["t_llm2"] for row in failure_log]
print(f"LLM time per call over these runs: mean {sum(llm_times) / len(llm_times):.2f} s "
      f"(min {min(llm_times):.2f} s, max {max(llm_times):.2f} s)")

Rank of the gold chunk (the chunk that contains the true answer)

category                     gold chunk  bi-encoder  cross-encoder  in P1 top-3  in P2 top-3
Keyword trap                       [40]          #7             #1           NO          YES
Exception (penalty)                [22]          #8             #1           NO          YES
Exception (except-clause)          [21]          #5             #1           NO          YES
Negation                          [118]          #8             #2           NO          YES
Buried chunk (numbers)             [47]          #7             #2           NO          YES
Limit: gold outside top-10        [109]         #16  not in top-10           NO           NO


Answer 1 vs Answer 2

1. [Keyword trap] What is the maximum number of working days of leave a student can take in a semester?
   True answer: 18 working days per semester (Sec 3.6, PDF p.19). The 6-day cap is only a proposal that is on hold (Sec 3.18).
   Answer 1 (baseline):  The

## Step 8: Latency profiling
Retrieval (Pipeline 1) and retrieval + cross-encoder re-ranking (Pipeline 2) are timed over 20 runs each (after 1 warm-up run) for 8 queries.
The LLM call is excluded here: it is the same for both pipelines (see the mean LLM time printed in Step 7) and repeating it would only use API quota.
A scaling experiment then times the cross-encoder on k = 10, 50, 100 and all 266 chunks.

In [23]:
import statistics

N_RUNS = 20   # repeat each measurement and average, so one slow run does not distort the result

# The 6 failure queries + the 2 sanity queries from Steps 4-5
latency_queries = [q for _, q, _, _ in FAILURE_QUERIES] + [
    "What is the maximum amount the Student Medical Reimbursement Fund pays per person?",
    "How many confirmed cases of Hand, Foot, and Mouth disease were recorded?",
]

def time_ms(fn, n=N_RUNS):
    """Run fn once to warm up, then n times; return (mean, standard deviation) in milliseconds."""
    fn()
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return statistics.mean(times), statistics.stdev(times)

# Time only the retrieval stages: no LLM calls here, so no Gemini quota is used
latency_rows = []
for q in latency_queries:
    candidates = retrieve(q)
    ret_mean, ret_sd = time_ms(lambda: retrieve(q))                # Pipeline 1: encode query + FAISS search
    rr_mean, rr_sd = time_ms(lambda: rerank(q, candidates))        # extra step in Pipeline 2: 10 cross-encoder pairs
    latency_rows.append((q, ret_mean, ret_sd, rr_mean, rr_sd))

print(f"Device: {device} | {N_RUNS} timed runs per measurement (after 1 warm-up run)\n")
print(f"{'query':<48} {'P1 retrieval':>13} {'re-rank (10)':>13} {'P2 total':>9} {'P2/P1':>6}")
for q, rm, rs, cm, cs in latency_rows:
    print(f"{q[:47]:<48} {rm:7.1f} ±{rs:4.1f} {cm:7.1f} ±{cs:4.1f} {rm + cm:9.1f} {(rm + cm) / rm:5.1f}x")

avg_ret = statistics.mean(r[1] for r in latency_rows)
avg_rr = statistics.mean(r[3] for r in latency_rows)
print(f"\n{'AVERAGE (ms)':<48} {avg_ret:13.1f} {avg_rr:13.1f} {avg_ret + avg_rr:9.1f} {(avg_ret + avg_rr) / avg_ret:5.1f}x")

Device: cuda | 20 timed runs per measurement (after 1 warm-up run)

query                                             P1 retrieval  re-rank (10)  P2 total  P2/P1
What is the maximum number of working days of l      6.5 ± 0.2    28.2 ± 4.3      34.6   5.4x
What happens to a student who backs out of an a      6.5 ± 0.2    24.9 ± 1.5      31.3   4.9x
How many days of leave can a student get for re      6.4 ± 0.4    25.8 ± 1.9      32.2   5.0x
Has any water softener been installed in the ho      6.1 ± 0.2    26.6 ± 1.4      32.7   5.3x
What is the current monthly stipend for a Quart      6.4 ± 0.2    25.0 ± 1.4      31.4   4.9x
Was weekly room cleaning introduced in the host      6.3 ± 0.6    26.1 ± 1.1      32.4   5.1x
What is the maximum amount the Student Medical       6.4 ± 0.3    25.8 ± 1.8      32.2   5.0x
How many confirmed cases of Hand, Foot, and Mou      6.5 ± 0.2    26.6 ± 1.3      33.1   5.1x

AVERAGE (ms)                                               6.4          26.1      32.

In [24]:
# Scaling experiment: what if the cross-encoder scored ALL chunks instead of only the bi-encoder's top-10?
all_candidates = [(c["chunk_id"], 0.0) for c in chunks]

print(f"{'k (pairs scored)':>16} {'cross-encoder time':>19} {'per pair':>9}")
for k in [10, 50, 100, len(chunks)]:
    q = latency_queries[0]
    cand_k = retrieve(q, k=k)
    mean_ms, _ = time_ms(lambda: rerank(q, cand_k), n=5)
    print(f"{k:>16} {mean_ms:16.1f} ms {mean_ms / k:7.2f} ms")

# Does scoring every chunk find the answer that the bi-encoder missed (Limit query, gold at bi-encoder rank #16)?
category, q, phrase, truth = FAILURE_QUERIES[5]
gold = find_gold(phrase)
ranked_all = rerank(q, all_candidates)
pos = {cid: r for r, (cid, _, _) in enumerate(ranked_all, start=1)}
print(f"\n[{category}] {q}")
print(f"Gold chunk {gold}: cross-encoder rank over ALL {len(chunks)} chunks = #{min(pos[g] for g in gold)}")

k (pairs scored)  cross-encoder time  per pair
              10             24.5 ms    2.45 ms
              50            122.4 ms    2.45 ms
             100            306.7 ms    3.07 ms
             266            857.8 ms    3.22 ms

[Limit: gold outside top-10] Was weekly room cleaning introduced in the hostels?
Gold chunk [109]: cross-encoder rank over ALL 266 chunks = #2
